In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
from scipy.io import loadmat
from PIL import Image
from transformers import ViTModel, ViTConfig
from tqdm import tqdm
import random

In [ ]:
# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 设置随机种子为42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 设置随机种子
set_seed(42)

# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 图像数据集路径
image_root = "L:/常惠林/萎凋/自然萎凋/原始"

# 图像预处理
transform = transforms.Compose([
    transforms.RandomRotation(degrees=30),  # 随机旋转图像，最大旋转角度为30度
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # 随机裁剪图像并调整为224x224，裁剪比例在0.8到1.0之间
    transforms.GaussianBlur(5),  # 对图像应用高斯模糊，核大小为5
    transforms.RandomHorizontalFlip(),  # 随机水平翻转图像
    transforms.ToTensor(),  # 将图像转换为张量，并将像素值归一化到[0, 1]
])

In [ ]:
# 修改MultimodalDataset类以支持NIR数据标准化
class MultimodalDataset(Dataset):
    def __init__(self, image_root, nir_path, transform=None, normalize_nir=True):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.normalize_nir = normalize_nir

        for i, cls in enumerate(["class_1", "class_2", "class_3"]):
            folder = os.path.join(image_root, cls)
            for fname in sorted(os.listdir(folder)):
                self.image_paths.append(os.path.join(folder, fname))
                self.labels.append(i)

        # 加载NIR数据
        nir_data = loadmat(nir_path)['nir']  # 假设键为'nir'
        self.nir_data = torch.tensor(nir_data, dtype=torch.float32)
        
        # NIR数据标准化
        if normalize_nir:
            self.nir_mean = torch.mean(self.nir_data, dim=0)
            self.nir_std = torch.std(self.nir_data, dim=0)
            # 避免除零错误
            self.nir_std = torch.where(self.nir_std == 0, torch.ones_like(self.nir_std), self.nir_std)
            self.nir_data = (self.nir_data - self.nir_mean) / self.nir_std

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        nir = self.nir_data[idx]
        label = self.labels[idx]
        return image, nir, label

# 简单的1D CNN用于NIR编码器
class NIREncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, L)
        x = self.net(x)
        return x.squeeze(-1)
# 对比学习模块
class ContrastiveLearning(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        
    def forward(self, features):
        # features: [2*N, D] - 同一批次的两种增强视图
        batch_size = features.shape[0] // 2
        labels = torch.arange(batch_size, device=features.device)
        labels = torch.cat([labels, labels], dim=0)
        
        # 计算相似度矩阵
        similarity_matrix = F.cosine_similarity(features.unsqueeze(1), features.unsqueeze(0), dim=2) / self.temperature
        
        # 正样本对的掩码
        mask = torch.zeros_like(similarity_matrix)
        for i in range(batch_size*2):
            for j in range(batch_size*2):
                if i != j and labels[i] == labels[j]:
                    mask[i, j] = 1
        
        # 移除对角线
        mask_diag = torch.ones_like(similarity_matrix) - torch.eye(batch_size*2, device=features.device)
        similarity_matrix = similarity_matrix * mask_diag
        
        # 计算对比损失
        positives = (similarity_matrix * mask).sum(dim=1) / mask.sum(dim=1)
        negatives = (similarity_matrix * (1 - mask)).sum(dim=1) / ((1 - mask).sum(dim=1))
        loss = -(positives - negatives).mean()
        return loss

# 掩码重建模块（适用于近红外数据）
class MaskedReconstructionModule(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
        
    def forward(self, x, mask_ratio=0.3):
        # x: [B, L] - 近红外光谱数据
        batch_size, length = x.shape
        
        # 创建掩码
        mask = torch.rand(batch_size, length, device=x.device) < mask_ratio
        x_masked = x.clone()
        x_masked[mask] = 0
        
        # 编码和解码
        features = self.encoder(x_masked)
        reconstructed = self.decoder(features)
        
        # 只计算被掩码部分的重建损失
        loss = F.mse_loss(reconstructed[mask], x[mask])
        return loss, features

# 多模态对齐模块
class MultimodalAlignmentModule(nn.Module):
    def __init__(self, img_dim, nir_dim, projection_dim=128):
        super().__init__()
        self.img_projection = nn.Linear(img_dim, projection_dim)
        self.nir_projection = nn.Linear(nir_dim, projection_dim)
        
    def forward(self, img_features, nir_features):
        # 投影到共同空间
        img_proj = F.normalize(self.img_projection(img_features), dim=1)
        nir_proj = F.normalize(self.nir_projection(nir_features), dim=1)
        
        # 计算图像和NIR表示之间的互信息损失
        similarity = torch.mm(img_proj, nir_proj.t())
        labels = torch.arange(similarity.shape[0], device=similarity.device)
        
        # 对称损失
        loss_i2n = F.cross_entropy(similarity, labels)
        loss_n2i = F.cross_entropy(similarity.t(), labels)
        loss = (loss_i2n + loss_n2i) / 2
        
        return loss

In [ ]:
# 多模态分类器
class MultimodalClassifier(nn.Module):
    def __init__(self, img_encoder, nir_encoder, hidden_dim=768):
        super().__init__()
        self.img_encoder = img_encoder
        self.nir_encoder = nir_encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 32, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, img, nir):
        img_feat = self.img_encoder(pixel_values=img).last_hidden_state[:, 0, :]  # CLS token
        nir_feat = self.nir_encoder(nir)
        fused = torch.cat([img_feat, nir_feat], dim=1)
        return self.classifier(fused), img_feat


In [ ]:
# 改进训练函数，增加梯度裁剪
def train_epoch(model, teacher, dataloader, criterion_cls, criterion_kd, optimizer, clip_value=1.0):
    model.train()
    total_loss, total_correct = 0, 0
    
    for img, nir, label in tqdm(dataloader):
        img, nir, label = img.to(device), nir.to(device), label.to(device)

        with torch.no_grad():
            teacher_feat = teacher(pixel_values=img).last_hidden_state[:, 0, :]

        out, student_feat = model(img, nir)
        loss_cls = criterion_cls(out, label)
        loss_kd = criterion_kd(student_feat, teacher_feat)
        loss = loss_cls + 0.5 * loss_kd

        optimizer.zero_grad()
        loss.backward()
        
        # 添加梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        
        optimizer.step()

        total_loss += loss.item() * label.size(0)
        total_correct += (out.argmax(1) == label).sum().item()

    return total_loss / len(dataloader.dataset), total_correct / len(dataloader.dataset)

In [ ]:
# 评估函数改进 - 返回评估指标
def evaluate(model, dataloader, return_metrics=False):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    
    with torch.no_grad():
        for img, nir, label in dataloader:
            img, nir = img.to(device), nir.to(device)
            out, _ = model(img, nir)
            probs = F.softmax(out, dim=1)
            y_scores.append(probs.cpu().numpy())
            y_pred.append(probs.argmax(1).cpu().numpy())
            y_true.append(label.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_scores = np.concatenate(y_scores)
    
    # 计算整体准确率
    accuracy = (y_true == y_pred).mean()
    
    # 如果需要可视化，画出混淆矩阵和ROC曲线
    if not return_metrics:
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plt.title("Confusion Matrix")
        plt.show()

        # ROC曲线
        plt.figure()
        for i in range(3):
            fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.show()
    
    # 返回评估指标
    if return_metrics:
        from sklearn.metrics import precision_score, recall_score, f1_score
        
        # 计算每个类别的精确率、召回率和F1分数
        precision = precision_score(y_true, y_pred, average=None)
        recall = recall_score(y_true, y_pred, average=None)
        f1 = f1_score(y_true, y_pred, average=None)
        
        # 计算宏平均和微平均
        macro_precision = precision_score(y_true, y_pred, average='macro')
        macro_recall = recall_score(y_true, y_pred, average='macro')
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        
        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'macro_f1': macro_f1
        }
        return metrics
    
    return accuracy

# 自监督学习预训练函数
def pretrain_self_supervised(img_encoder, nir_encoder, dataloader, epochs=10):
    img_encoder.train()
    nir_encoder.train()
    
    # 初始化自监督模块
    contrastive_loss = ContrastiveLearning().to(device)
    masked_recon = MaskedReconstructionModule(input_dim=400).to(device)  # 假设NIR数据维度为400
    multimodal_align = MultimodalAlignmentModule(
        img_dim=384,  # ViT特征维度
        nir_dim=32    # NIR编码器输出维度
    ).to(device)
    
    # 图像增强转换
    img_transform1 = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip()
    ])
    
    img_transform2 = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
    ])
    
    # 优化器
    optimizer = torch.optim.Adam(
        list(img_encoder.parameters()) + 
        list(nir_encoder.parameters()) +
        list(masked_recon.parameters()) +
        list(multimodal_align.parameters()),
        lr=1e-4
    )
    
    for epoch in range(epochs):
        total_loss = 0
        for img, nir, _ in tqdm(dataloader, desc=f"预训练 Epoch {epoch+1}/{epochs}"):  # 忽略标签
            img, nir = img.to(device), nir.to(device)
            
            # 1. 图像对比学习
            img1 = img_transform1(img)
            img2 = img_transform2(img)
            imgs = torch.cat([img1, img2], dim=0)
            
            # 提取图像特征
            with torch.no_grad():
                img_features = img_encoder(pixel_values=imgs).last_hidden_state[:, 0, :]
            
            contrastive_img_loss = contrastive_loss(img_features)
            
            # 2. NIR掩码重建损失
            nir_recon_loss, nir_features = masked_recon(nir)
            
            # 3. 多模态对齐损失
            orig_img_features = img_encoder(pixel_values=img).last_hidden_state[:, 0, :]
            nir_encoded = nir_encoder(nir)
            alignment_loss = multimodal_align(orig_img_features, nir_encoded)
            
            # 总损失
            loss = contrastive_img_loss + nir_recon_loss + alignment_loss * 0.5
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")
    
    return img_encoder, nir_encoder

In [ ]:
# 主流程 - 修改为使用6:2:2数据集划分
if __name__ == '__main__':
    # 创建完整数据集
    full_dataset = MultimodalDataset(image_root, 'L:/常惠林/萎凋/NIR.mat', transform=transform)
    
    # 计算数据集划分长度
    dataset_size = len(full_dataset)
    train_size = int(dataset_size * 0.6)
    val_size = int(dataset_size * 0.2)
    test_size = dataset_size - train_size - val_size
    
    # 划分数据集
    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, 
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)  # 确保划分可重复
    )
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    print(f"数据集划分: 训练集 {len(train_dataset)}样本, 验证集 {len(val_dataset)}样本, 测试集 {len(test_dataset)}样本")
    
    # 初始化模型
    teacher_vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
    student_vit_config = ViTConfig(
        hidden_size=384,
        num_hidden_layers=6,
        num_attention_heads=6,
        intermediate_size=384*4,
        image_size=224,
        patch_size=16
    )
    student_vit = ViTModel(student_vit_config).to(device)
    nir_encoder = NIREncoder().to(device)
    
    # 1. 自监督预训练阶段 - 使用训练集
    print("开始自监督预训练...")
    student_vit, nir_encoder = pretrain_self_supervised(
        student_vit, 
        nir_encoder, 
        train_loader,  # 只在训练集上预训练
        epochs=5
    )
    print("自监督预训练完成！")
    
    # 2. 知识蒸馏微调阶段 - 使用训练集训练，验证集评估
    model = MultimodalClassifier(student_vit, nir_encoder, hidden_dim=384).to(device)
    
    criterion_cls = nn.CrossEntropyLoss()
    criterion_kd = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)  # 添加权重衰减
    
    # 添加学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    print("开始知识蒸馏微调...")
    best_val_acc = 0
    patience = 3
    patience_counter = 0
    
    for epoch in range(20):  # 增加训练轮数
        # 训练
        train_loss, train_acc = train_epoch(model, teacher_vit, train_loader, criterion_cls, criterion_kd, optimizer)
        
        # 验证
        val_metrics = evaluate(model, val_loader, return_metrics=True)
        val_acc = val_metrics['accuracy']
        
        # 更新学习率
        scheduler.step(val_acc)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # 早停策略
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            # 保存最佳模型
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'epoch': epoch,
                'val_acc': val_acc,
                'train_acc': train_acc,
                'train_loss': train_loss,
            }, 'best_multimodal_model.pt')
            print(f"保存新的最佳模型，验证准确率: {val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # 加载最佳模型进行测试
    checkpoint = torch.load('best_multimodal_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 3. 模型评估 - 使用测试集
    print("评估最终模型性能...")
    test_metrics = evaluate(model, test_loader)
    print(f"测试集准确率: {test_metrics:.4f}")